In [8]:
import bs4
from langsmith import Client
hub = Client()
from operator import itemgetter

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from chromadb.utils import embedding_functions
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.load import dumps, loads

from dotenv import load_dotenv


load_dotenv()


True

In [6]:

#### INDEXING ####

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

embedding_function = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Embed
vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=embedding_function)

retriever = vectorstore.as_retriever()

llm = ChatGroq(temperature=0, model='llama-3.3-70b-versatile')



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Multi Query

In [10]:

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser

generate_queries = (
    prompt_perspectives 
    | llm
    | StrOutputParser() 
    | (lambda x: x.split("\n\n"))
)

In [11]:

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

# Retrieve
question = "What is task decomposition for LLM agents?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question":question})
len(docs)

/tmp/ipykernel_11435/3045963154.py:8: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]
/tmp/ipykernel_11435/3045963154.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


8

In [12]:
retriever.map()

RunnableEach(bound=VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x79d919e90800>, search_kwargs={}))

In [13]:
(generate_queries | retriever.map()).invoke({"question": question})

[[Document(id='06dc7a4e-533f-4126-8503-8671bf97fdbf', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Or\n@article{weng2023agent,\n  title   = "LLM-powered Autonomous Agents",\n  author  = "Weng, Lilian",\n  journal = "lilianweng.github.io",\n  year    = "2023",\n  month   = "Jun",\n  url     = "https://lilianweng.github.io/posts/2023-06-23-agent/"\n}\nReferences#\n[1] Wei et al. “Chain of thought prompting elicits reasoning in large language models.” NeurIPS 2022\n[2] Yao et al. “Tree of Thoughts: Dliberate Problem Solving with Large Language Models.” arXiv preprint arXiv:2305.10601 (2023).\n[3] Liu et al. “Chain of Hindsight Aligns Language Models with Feedback\n“ arXiv preprint arXiv:2302.02676 (2023).\n[4] Liu et al. “LLM+P: Empowering Large Language Models with Optimal Planning Proficiency” arXiv preprint arXiv:2304.11477 (2023).\n[5] Yao et al. “ReAct: Synergizing reasoning and acting in language models.” ICLR 2023.\n[6] Google Blog. “An

In [14]:
from operator import itemgetter

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

llm = ChatGroq(temperature=0, model='llama-3.3-70b-versatile')

final_rag_chain = (
    {"context": retrieval_chain, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

/tmp/ipykernel_11435/3045963154.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, more manageable subgoals or steps. This allows the agent to efficiently handle complex tasks by dividing them into simpler, more manageable components. Task decomposition can be achieved through various methods, including:\n\n1. Simple prompting: Using prompts like "Steps for XYZ" or "What are the subgoals for achieving XYZ?" to guide the LLM in decomposing the task.\n2. Task-specific instructions: Providing specific instructions, such as "Write a story outline" for writing a novel.\n3. Human inputs: Receiving input from humans to help decompose the task.\n4. Chain of thought (CoT) prompting: Instructing the model to "think step by step" to decompose hard tasks into smaller and simpler steps.\n5. Tree of Thoughts (ToT): Exploring multiple reasoning possibilities at each step, creating a tree structure of thoughts and subgoals.\n6. LLM+P approach: Using an externa

# RAG-Fusion

In [15]:
template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [16]:


generate_queries = (
    prompt_rag_fusion 
    | llm
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

In [17]:
question = "What is task decomposition for LLM agents?"

In [18]:

def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents 
        and an optional parameter k used in the RRF formula """
    
    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

/tmp/ipykernel_11435/3798540717.py:24: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


12

In [19]:
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

/tmp/ipykernel_11435/3798540717.py:24: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down large tasks into smaller, manageable subgoals. This enables the agent to handle complex tasks more efficiently. Task decomposition can be done in several ways, including:\n\n1. Using simple prompting, such as "Steps for XYZ" or "What are the subgoals for achieving XYZ?"\n2. Using task-specific instructions, such as "Write a story outline" for writing a novel\n3. With human inputs\n4. Using an external classical planner, such as LLM+P, which utilizes the Planning Domain Definition Language (PDDL) to describe the planning problem and generate a plan.\n\nThe goal of task decomposition is to enable the LLM agent to efficiently handle complex tasks by breaking them down into smaller, more manageable subgoals, and then using reflection and refinement to learn from mistakes and improve the quality of the final results.'

# Decomposition

In [20]:
# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

In [24]:
# Chain
generate_queries_decomposition = ( prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n\n")))

# Run
question = "What are the main components of an LLM-powered autonomous agent system?"
questions = generate_queries_decomposition.invoke({"question":question})

In [25]:
questions

['To break down the question into manageable sub-questions, here are three search queries:',
 '1. **What are the key architectural components of an autonomous agent system?**\n   - This query focuses on understanding the overall structure and essential parts that make up an autonomous agent system, which can include perception, decision-making, and action components.',
 "2. **How does a Large Language Model (LLM) integrate with an autonomous agent's decision-making process?**\n   - This query delves into the specifics of how LLMs contribute to the decision-making aspect of an autonomous agent, including how they process inputs, generate outputs, and learn from interactions.",
 '3. **What role do perception and sensing systems play in an LLM-powered autonomous agent, and how do they interact with the LLM?**\n   - This query explores the input side of the autonomous agent, focusing on how the agent perceives its environment through various sensors and how this sensory information is proc

In [26]:
# Prompt
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)

In [28]:
def format_qa_pair(question, answer):
    """Format Q and A pair"""
    
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()


q_a_pairs = ""
for q in questions:
    
    rag_chain = (
    {"context": itemgetter("question") | retriever, 
     "question": itemgetter("question"),
     "q_a_pairs": itemgetter("q_a_pairs")} 
    | decomposition_prompt
    | llm
    | StrOutputParser())

    answer = rag_chain.invoke({"question":q,"q_a_pairs":q_a_pairs})
    q_a_pair = format_qa_pair(q,answer)
    q_a_pairs = q_a_pairs + "\n---\n"+  q_a_pair

In [30]:
print(answer)

To understand the role of perception and sensing systems in a LLM-powered autonomous agent and how they interact with the LLM, let's break down the key components and processes involved.

1. **Perception and Sensing Systems**: These systems are crucial for an autonomous agent as they enable the agent to gather information about its environment. This can include visual data from cameras, auditory data from microphones, tactile data from sensors, and other types of sensory inputs. The perception system processes this raw sensory data into a format that can be understood and utilized by the agent.

2. **Interaction with LLM**: The processed sensory information is then fed into the Large Language Model (LLM) as input. The LLM, acting as the agent's brain, processes this information to understand the current state of the environment. This understanding is essential for the agent to make informed decisions about its actions.

3. **LLM Processing**: The LLM analyzes the sensory information in

# Step Back

In [36]:
# Few Shot Examples
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "what is Jan Sindel’s personal history?",
    },
]
# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)

In [37]:
prompt

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:'), additional_kwargs={}), FewShotChatMessagePromptTemplate(examples=[{'input': 'Could the members of The Police perform lawful arrests?', 'output': 'what can the members of The Police do?'}, {'input': 'Jan Sindel’s was born in what country?', 'output': 'what is Jan Sindel’s personal history?'}], input_variables=[], input_types={}, partial_variables={}, example_prompt=ChatPromptTemplate(input_variables=['input', 'output'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}

In [38]:
generate_queries_step_back = prompt | llm | StrOutputParser()
question = "What is task decomposition for LLM agents?"
generate_queries_step_back.invoke({"question": question})

'What is task decomposition in general?'

In [39]:
from langchain_core.runnables import RunnableLambda


response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant.

# {normal_context}
# {step_back_context}

# Original Question: {question}
# Answer:"""
response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

chain = (
    {
        # Retrieve context using the normal question
        "normal_context": RunnableLambda(lambda x: x["question"]) | retriever,
        # Retrieve context using the step-back question
        "step_back_context": generate_queries_step_back | retriever,
        # Pass on the question
        "question": lambda x: x["question"],
    }
    | response_prompt
    | llm
    | StrOutputParser()
)

chain.invoke({"question": question})

'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, more manageable subgoals or steps. This is a crucial component of planning in LLM-powered autonomous agents, as it enables the agent to efficiently handle complex tasks and improve the quality of its final results.\n\nThere are several approaches to task decomposition for LLM agents, including:\n\n1. **Simple prompting**: The LLM can be prompted with simple instructions, such as "Steps for XYZ" or "What are the subgoals for achieving XYZ?", to generate a list of subgoals or steps.\n2. **Task-specific instructions**: The LLM can be provided with task-specific instructions, such as "Write a story outline" for writing a novel, to guide the decomposition process.\n3. **Human inputs**: Human inputs can be used to provide additional context or guidance for the task decomposition process.\n4. **LLM+P approach**: This approach involves relying on an external classical pl

# HyDE

In [41]:
# HyDE document generation
template = """Please write a scientific paper passage to answer the question
Question: {question}
Passage:"""
prompt_hyde = ChatPromptTemplate.from_template(template)

from langchain_core.output_parsers import StrOutputParser

generate_docs_for_retrieval = (
    prompt_hyde | llm | StrOutputParser() 
)

# Run
question = "What is task decomposition for LLM agents?"
generate_docs_for_retrieval.invoke({"question":question})

'**Task Decomposition for Large Language Model (LLM) Agents: A Framework for Efficient Problem-Solving**\n\nTask decomposition is a crucial technique employed by Large Language Model (LLM) agents to efficiently solve complex problems. At its core, task decomposition involves breaking down a complex task into a series of smaller, more manageable sub-tasks that can be executed sequentially or in parallel. This approach enables LLM agents to tackle intricate problems by dividing them into simpler, more tractable components, thereby reducing the computational complexity and improving overall performance.\n\nIn the context of LLM agents, task decomposition typically involves a hierarchical representation of the problem-solving process. The agent begins by analyzing the input task and identifying the key objectives, constraints, and requirements. The task is then decomposed into a set of sub-tasks, each of which is associated with a specific goal or objective. These sub-tasks are further ref

In [42]:
# Retrieve
retrieval_chain = generate_docs_for_retrieval | retriever 
retrieved_docs = retrieval_chain.invoke({"question":question})
retrieved_docs

[Document(id='21c6e847-c0f5-462e-b5bc-e633c6814fbc', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.\nAnother quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermediate interface to describe the planning problem. In this process, LLM (1) translates the problem into “Problem PDDL”, then (2) requests a classical planner to generate a PDDL plan based on an existing “Domain PDDL”, and finally (3) translates the PDDL plan back into natural language. Essentially, the planning step is outsourced to an external tool, assuming the availability of 

In [43]:
# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"context":retrieved_docs,"question":question})

'Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, more manageable steps. This can be achieved through various methods, including:\n\n1. Simple prompting: Using prompts like "Steps for XYZ" or "What are the subgoals for achieving XYZ?" to guide the LLM to decompose the task.\n2. Task-specific instructions: Providing specific instructions, such as "Write a story outline" for writing a novel.\n3. Human inputs: Utilizing human input to help decompose the task.\n4. Chain of Thought (CoT): Instructing the model to "think step by step" to utilize more test-time computation to decompose hard tasks into smaller and simpler steps.\n5. Tree of Thoughts: Exploring multiple reasoning possibilities at each step, creating a tree structure, and using search processes like breadth-first search (BFS) or depth-first search (DFS) to evaluate each state.\n6. LLM+P approach: Using an external classical planner to do long-horizon plan